# 48 · Your first evals
Level L5 · Part 6 · core · ~35 min · builds on 46

## Goal
Build a golden set of 10 Scout questions and a scorer, run every question 3 times to see that a real
model's answers vary from run to run, and compare the pass rate and pass@k of two system prompts.

## The idea
Lesson 46's tests check the loop's **mechanics** with a fake. They cannot tell you whether Scout gives
**good answers**. For that you need an **eval**: a fixed list of questions with known right answers (a
**golden set**), a **scorer** that marks each answer pass or fail, and one number at the end, the
**pass rate**. A real model samples its words, so one run proves little: you run each question several
times. **pass@k** asks "did at least one of k tries pass?"; **pass^k** asks "did all k pass?".

Analogy: a driving test. One lucky drive proves little; ten drives on the same route, scored by the
same checklist, tell you how good the driver is.

## Picture
```mermaid
flowchart LR
    G["golden set<br/>10 questions + expected facts"] --> R["run each question<br/>n = 3 times"]
    P["system prompt<br/>v1 or v2"] --> R
    R --> S["scorer<br/>pass / fail"]
    S --> M["pass rate, pass@1,<br/>pass@3, pass^3"]
    M --> C["compare v1 vs v2"]
```

## Step by step
### 1. Setup: the recorder, the model client and Scout's files
Every Gemini call is recorded once to `../data/llm_cassettes/48_first-evals.yaml` and replayed after
that, so the notebook runs with no key. Scout's loop, tools and fake come from `../apps/46/`, the files
lesson 46 wrote (run lesson 46 first if that folder is missing).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("48_first-evals")

import contextlib, io, json, os, random, re
from math import comb
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

sys.dont_write_bytecode = True                    # no __pycache__ in ../apps/46
assert Path("../apps/46/scout_loop.py").exists(), "run lesson 46 first: it writes ../apps/46/"
sys.path.insert(0, "../apps/46")
from fake_model import ScriptedModel
from scout_loop import run_agent
from scout_tools import NOTES, TOOLS

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
print("model:", MODEL, "| notes:", len(NOTES), "| tools:", list(TOOLS))

LLM mode: replay (48_first-evals.yaml)


model: gemini-3.5-flash-lite | notes: 6 | tools: ['search_notes', 'calculator']


### 2. The golden set
Each **case** has an id, a question, and a list of **expected facts**. A fact is a list of accepted
spellings: `["two", "2"]` means "two" or "2" must appear. Every answer comes from Scout's six notes,
sometimes with arithmetic: a Mars day of 24 h 39 min is 1479 minutes, 88740 seconds, and 10 such days
are 246.5 hours. Note the real-world figure differs slightly (about 24 h 39 min 35 s), so a model that
answers from memory instead of the notes can miss.

In [2]:
GOLDEN = [
    {"id": "moons-count",  "q": "How many moons does Mars have?",           "facts": [["two", "2"]]},
    {"id": "moon-names",   "q": "What are the names of Mars's moons?",      "facts": [["phobos"], ["deimos"]]},
    {"id": "backwards",    "q": "Which planet spins backwards?",            "facts": [["venus"]]},
    {"id": "largest",      "q": "Which planet is the largest?",             "facts": [["jupiter"]]},
    {"id": "rings",        "q": "What are Saturn's rings made of?",         "facts": [["ice"]]},
    {"id": "rover-date",   "q": "When did Perseverance land on Mars?",      "facts": [["february"], ["2021"]]},
    {"id": "day-length",   "q": "How long is a day on Mars?",               "facts": [["24 hours"], ["39 minutes"]]},
    {"id": "day-minutes",  "q": "How many minutes long is a day on Mars?",  "facts": [["1479"]]},
    {"id": "day-seconds",  "q": "How many seconds long is a day on Mars?",  "facts": [["88740"]]},
    {"id": "ten-days",     "q": "How many hours are 10 days on Mars?",      "facts": [["246.5", "246 hours and 30 minutes"]]},
]
print(len(GOLDEN), "cases;", sum(len(c["facts"]) for c in GOLDEN), "expected facts in total")

10 cases; 13 expected facts in total


### 3. The scorer
The scorer is plain code, so it is fast, free and gives the same verdict every time. It lowercases the
answer, drops thousands separators (`1,479` becomes `1479`), and needs **every** fact, matched as whole
words (`\b` in a regex), so `"2"` does not match inside `"12"`.

In [3]:
def normalise(text: str) -> str:
    return re.sub(r"(?<=\d),(?=\d{3})", "", text.lower())

def has(text: str, phrase: str) -> bool:
    return re.search(rf"\b{re.escape(phrase.lower())}\b", text) is not None

def score(answer: str | None, facts: list[list[str]]) -> bool:
    text = normalise(answer or "")
    return all(any(has(text, alt) for alt in alts) for alts in facts)

print(score("A day on Mars is 1,479 minutes.", [["1479"]]),
      score("Mars has 12 moons.", [["two", "2"]]),
      score("Phobos only.", [["phobos"], ["deimos"]]))

True False False


`True False False`: the comma was ignored, `2` did not match inside `12`, and a half answer failed.
Test the scorer on hand-written answers like this before you trust any number it gives.

### 4. The eval harness, tried on a noisy fake
`run_eval` runs every case `n` times with a **fresh** model, scores each answer and keeps the answers
(you will want to read them). The loop's per-step printing is hidden. To see the harness work without a
key, we use a fake that answers right 70% of the time, with a fixed random seed.

In [4]:
def run_eval(make_model, cases=GOLDEN, n=3, schemas=None) -> dict:
    results = {}
    for case in cases:
        runs = []
        for _ in range(n):
            with contextlib.redirect_stdout(io.StringIO()):          # hide the step trace
                out = run_agent(case["q"], make_model(case), TOOLS, schemas or [], max_steps=5)
            runs.append({"passed": score(out["answer"], case["facts"]), "answer": out["answer"]})
        results[case["id"]] = runs
    return results

rng = random.Random(48)
def noisy_fake(case):                                  # right 70% of the time, wrong otherwise
    right = " and ".join(alts[0] for alts in case["facts"])
    return ScriptedModel([lambda m: {"text": right if rng.random() < 0.7 else "I am not sure."}])

fake = run_eval(noisy_fake)
print({cid: "".join("P" if r["passed"] else "." for r in runs) for cid, runs in fake.items()})

{'moons-count': 'PP.', 'moon-names': 'P.P', 'backwards': '.P.', 'largest': '.PP', 'rings': 'PP.', 'rover-date': 'PPP', 'day-length': 'PPP', 'day-minutes': 'PPP', 'day-seconds': 'PP.', 'ten-days': 'PP.'}


`P` is a pass, `.` a fail, one character per run. The same question, asked three times, passed
sometimes and failed sometimes. A real model samples its words too, so its answers vary run to run;
with this golden set the real model's pass/fail happened not to flip (step 7), which is why we use the
fake to show what pass@k and pass^k do.

### 5. Pass rate and pass@k
With `n` runs of a case and `c` passes: **pass rate** = c / n over all runs. **pass@k** is the chance
that at least one of `k` runs, picked from the `n`, passes; **pass^k** is the chance that all `k` pass.
The formulas count combinations, so they are exact for the runs you have.

In [5]:
def pass_at_k(n: int, c: int, k: int) -> float:
    return 1.0 if n - c < k else 1 - comb(n - c, k) / comb(n, k)

def pass_hat_k(n: int, c: int, k: int) -> float:
    return comb(c, k) / comb(n, k)

def summary(results: dict, k: int = 3) -> dict:
    counts = [(len(runs), sum(r["passed"] for r in runs)) for runs in results.values()]
    mean = lambda xs: round(sum(xs) / len(xs), 2)
    return {"pass rate": mean([c / n for n, c in counts]),
            "pass@1": mean([pass_at_k(n, c, 1) for n, c in counts]),
            f"pass@{k}": mean([pass_at_k(n, c, k) for n, c in counts]),
            f"pass^{k}": mean([pass_hat_k(n, c, k) for n, c in counts])}

print("noisy fake:", summary(fake))

noisy fake: {'pass rate': 0.73, 'pass@1': 0.73, 'pass@3': 1.0, 'pass^3': 0.3}


On the fake: pass@1 equals the pass rate. pass@3 is much higher: if you may retry and a person picks
the good answer, most questions get one. pass^3 is much lower: a user who asks three times gets three
right answers far less often. For an agent that runs unattended, pass^k is the number that hurts. A
real agent's pass^k can be this far below pass@k whenever its per-run success rate is below 1.

### 6. The real model behind the loop (setup from lesson 35)
Gemini needs full tool schemas, not the short `SCHEMAS` in `../apps/46/`. Results go back under
`"output"` or `"error"`, and each model turn goes back unchanged (it carries a thought signature).

In [6]:
FULL_SCHEMAS = [
    {"name": "search_notes", "description": TOOLS["search_notes"].__doc__,
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "calculator", "description": TOOLS["calculator"].__doc__,
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
]

def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                            # keeps the thought signature
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The model wrapper takes the **system prompt** as an argument, since that is what we will change. It
keeps Gemini's default `temperature=1.0`, written out so you can see it: the eval should measure the
model as users will meet it, not a frozen version of it.

In [7]:
class GeminiModel:
    def __init__(self, system: str):
        self.system = system

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.Tool(function_declarations=[types.FunctionDeclaration(
            name=s["name"], description=s["description"], parameters_json_schema=s["parameters"]) for s in tools])]
        cfg = types.GenerateContentConfig(system_instruction=self.system, temperature=1.0, max_output_tokens=300,
                                          tools=decls, automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        raw = resp.candidates[0].content
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        if calls:
            return {"role": "assistant", "tool_calls": calls, "raw": raw}
        return {"role": "assistant", "text": resp.text, "raw": raw}

### 7. Prompt v1 on the golden set
Version 1 of the system prompt is short. It does not say where answers should come from. 10 questions
times 3 runs is 30 agent runs. If the cell is skipped (no key, no recording), `real` stays empty and the
cells after it say so instead of failing.

In [8]:
SYSTEM_V1 = "You are Scout, a helpful assistant. Answer in one sentence."
real = {}
print("v1:", SYSTEM_V1)

v1: You are Scout, a helpful assistant. Answer in one sentence.


Now the 30 runs with v1. Each answer is scored as it arrives; the cell prints one `P`/`.` string per case.

In [9]:
real["v1"] = run_eval(lambda case: GeminiModel(SYSTEM_V1), schemas=FULL_SCHEMAS)
print({cid: "".join("P" if r["passed"] else "." for r in runs) for cid, runs in real["v1"].items()})
print("v1:", summary(real["v1"]))

{'moons-count': 'PPP', 'moon-names': 'PPP', 'backwards': 'PPP', 'largest': 'PPP', 'rings': 'PPP', 'rover-date': 'PPP', 'day-length': 'PPP', 'day-minutes': '...', 'day-seconds': '...', 'ten-days': '...'}
v1: {'pass rate': 0.7, 'pass@1': 0.7, 'pass@3': 0.7, 'pass^3': 0.7}


21 of 30 runs passed: a pass rate of 0.7. The seven note facts passed every run. The three arithmetic
cases (minutes, seconds, 10 days) failed every run. Here each case passed all three runs or none, so
pass@3 and pass^3 equal the pass rate. The next step shows why they failed.

### 8. Read the failures
A number says **that** Scout failed, not **why**. Print every failing answer next to its expected facts.
This is how you find a broken scorer, a bad golden answer or a real model mistake.

In [10]:
def failures(results: dict) -> None:
    facts = {c["id"]: c["facts"] for c in GOLDEN}
    for cid, runs in results.items():
        for i, r in enumerate(runs, 1):
            if not r["passed"]:
                print(f"{cid} run {i}: expected {facts[cid]}\n   got: {(r['answer'] or '(no answer)').strip()[:150]}")

failures(real["v1"]) if "v1" in real else print("v1 was skipped: no recording and no key")

day-minutes run 1: expected [['1479']]
   got: A day on Mars, known as a sol, is approximately 1,475 minutes long (or 24 hours and 39 minutes).
day-minutes run 2: expected [['1479']]
   got: A day on Mars, known as a sol, is approximately 1,478 minutes long.
day-minutes run 3: expected [['1479']]
   got: A day on Mars, known as a sol, is approximately 1,477 minutes long.
day-seconds run 1: expected [['88740']]
   got: A day on Mars, known as a sol, is approximately 88,775 seconds long.
day-seconds run 2: expected [['88740']]
   got: A day on Mars, known as a sol, is approximately 88,775 seconds long.
day-seconds run 3: expected [['88740']]
   got: A day on Mars, known as a sol, is approximately 88,775.24 seconds long.
ten-days run 1: expected [['246.5', '246 hours and 30 minutes']]
   got: Ten days on Mars are approximately 246.6 hours, as a Martian sol is about 24 hours and 39 minutes long.
ten-days run 2: expected [['246.5', '246 hours and 30 minutes']]
   got: Ten days on Mars are a

The answers vary run to run: 1,475, 1,478 and 1,477 minutes for the same question. That is sampling
at `temperature=1.0`. The seconds answer, 88,775, and 246.6 hours are the real-world figures (a sol of
about 24 h 39 min 35 s), not the notes' 24 h 39 min. The scorer is right and the golden answers are
right: v1 never says to answer from the notes, so Scout answered from memory. Pass/fail did not change
between runs here, but the answers did; that is the non-determinism the fake in step 4 exaggerates so
you can see pass@k.

### 9. Prompt v2: the same golden set, a changed prompt
Version 2 tells Scout to search the notes first, answer only from them and use the calculator. Same
questions, same scorer, same 3 runs each. Only the prompt changed.

In [11]:
SYSTEM_V2 = ("You are Scout. Answer questions about the user's notes. Always search the notes first and "
             "answer only from what they say. Use the calculator for any arithmetic. Answer in one sentence.")
print("v2:", SYSTEM_V2)

v2: You are Scout. Answer questions about the user's notes. Always search the notes first and answer only from what they say. Use the calculator for any arithmetic. Answer in one sentence.


The same 30 runs with v2, in the same order and with the same scorer.

In [12]:
real["v2"] = run_eval(lambda case: GeminiModel(SYSTEM_V2), schemas=FULL_SCHEMAS)
print({cid: "".join("P" if r["passed"] else "." for r in runs) for cid, runs in real["v2"].items()})
print("v2:", summary(real["v2"]))

{'moons-count': 'PPP', 'moon-names': 'PPP', 'backwards': 'PPP', 'largest': 'PPP', 'rings': 'PPP', 'rover-date': 'PPP', 'day-length': 'PPP', 'day-minutes': 'PPP', 'day-seconds': 'PPP', 'ten-days': 'PPP'}
v2: {'pass rate': 1.0, 'pass@1': 1.0, 'pass@3': 1.0, 'pass^3': 1.0}


### 10. Before and after
One table: each case's passes out of 3 for each prompt, then the summary numbers side by side.

In [13]:
if {"v1", "v2"} <= real.keys():
    print(f"{'case':13} v1   v2")
    for cid in real["v1"]:
        v1, v2 = (sum(r["passed"] for r in real[v][cid]) for v in ("v1", "v2"))
        print(f"{cid:13} {v1}/3  {v2}/3{'   <- changed' if v1 != v2 else ''}")
    for metric in summary(real["v1"]):
        print(f"{metric:10} {summary(real['v1'])[metric]:5}  {summary(real['v2'])[metric]:5}")
else:
    print("skipped: run with a recording or a key to compare v1 and v2")

case          v1   v2
moons-count   3/3  3/3
moon-names    3/3  3/3
backwards     3/3  3/3
largest       3/3  3/3
rings         3/3  3/3
rover-date    3/3  3/3
day-length    3/3  3/3
day-minutes   0/3  3/3   <- changed
day-seconds   0/3  3/3   <- changed
ten-days      0/3  3/3   <- changed
pass rate    0.7    1.0
pass@1       0.7    1.0
pass@3       0.7    1.0
pass^3       0.7    1.0


v2 passed all 30 runs: the three arithmetic cases went from 0/3 to 3/3 and nothing else moved. One
eval run is still a small sample: 10 cases and 3 runs each. Before trusting a change, grow the golden
set and add cases the new prompt might break, such as a question the notes do not answer.

## At work
- **Start small, from real failures.** Anthropic's "Demystifying evals for AI agents" says "20-50 simple
  tasks drawn from real failures is a great start" (10 cases here; start at 20-50). Logged bad
  production runs become new golden cases → lesson 49.
- **Code graders first.** The same article recommends "deterministic graders where possible, LLM graders
  where necessary".
- **Two kinds of eval.** Capability evals start at a low pass rate and show progress; regression evals
  should stay near 100% and block a release that breaks them.
- **Run the eval before every swap.** Prompt edits and model upgrades are both changes; pin the model id
  and rerun the golden set before switching → lesson 53.

## What just happened
- A golden set of 10 Scout questions, each with expected facts, and a code scorer that matches whole
  words and ignores thousands separators.
- A harness that runs each case 3 times with a fresh model and keeps every answer.
- A seeded noisy fake showed the same question passing and failing across runs, and how pass@1, pass@3
  and pass^3 differ.
- Gemini at `temperature=1.0` gave different numbers for the same question (1,475, 1,478, 1,477
  minutes). With prompt v1 it answered arithmetic questions from memory: pass rate 0.7.
- Prompt v2 (search the notes first, use the calculator) passed all 30 runs: 1.0. The table showed the
  three cases that moved.

## Common mistakes
**1. A plain substring check.** `"2" in answer` looks fine, but it also passes answers that contain
"12" or "2021". The eval then reports a pass rate higher than the truth.

In [14]:
def loose_score(answer, facts):
    return all(any(alt in answer.lower() for alt in alts) for alts in facts)

wrong = "Mars has 12 moons."
assert not loose_score(wrong, [["two", "2"]]), f"loose scorer passed a wrong answer: {wrong!r}"

AssertionError: loose scorer passed a wrong answer: 'Mars has 12 moons.'

Fix: match whole words, as `score` does with `\b`, and test the scorer on wrong answers too.

In [15]:
print("strict scorer on the wrong answer:", score(wrong, [["two", "2"]]))

strict scorer on the wrong answer: False


**2. One run per question.** With one run, a lucky or unlucky sample decides the verdict. Here the
noisy fake is scored on its **first** run only, then on all three: the numbers disagree.

In [16]:
first_only = {cid: runs[:1] for cid, runs in fake.items()}
one, three = summary(first_only, k=1)["pass rate"], summary(fake)["pass rate"]
assert one == three, f"1 run says {one}, 3 runs say {three}: one run is not enough to decide"

AssertionError: 1 run says 0.8, 3 runs say 0.73: one run is not enough to decide

Fix: run every case several times (3 is a start; more for close calls) and compare pass rates over all
runs, as `run_eval(n=3)` does.

In [17]:
print("pass rate over 3 runs per case:", summary(fake)["pass rate"])

pass rate over 3 runs per case: 0.73


## Production checklist
- [ ] A golden set in version control, grown from real failures; each case has expected facts.
- [ ] A code scorer, itself tested on right **and** wrong answers.
- [ ] Each case run several times; report pass rate plus pass@k or pass^k, whichever matches the use.
- [ ] Failing answers read by a person after every eval run, not just the number.
- [ ] The eval runs before every prompt change and every model swap, with the model id pinned.
- [ ] The model is called with production settings (same temperature, prompt and tools).

## Try it
Some wrong answers contain a telltale fact: a Mars day in seconds from memory is about **88775**, not
the notes' 88740. Extend the scorer: `score_v2(answer, facts, forbidden=[])` fails if any forbidden
phrase appears (whole words). Add `"forbidden": ["88775"]` to the `day-seconds` case in a copy of the
golden set, and check it on `"About 88,775 seconds, or 88740 by my notes."` (fail) and
`"A Mars day is 88,740 seconds."` (pass).

In [18]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [19]:
llm_replay.stop()

## Key terms
- **golden set** — a fixed list of inputs with known right answers, kept in version control and reused for every eval run.
- **scorer** — code (or a model, or a person) that marks one output pass or fail against the golden answer.
- **pass rate** — the share of all runs that passed.
- **repeated runs** — running each eval case several times, because a sampling model can answer the same question differently each time.
- **pass@k** — the chance that at least one of k runs of a case passes.
- **pass^k** — the chance that all k runs of a case pass; the stricter number for agents that run unattended.